# 자율주행 파이프라인 (Ultra96V2 + DPU)

**실행 흐름:** 환경 설정 → 컴포넌트 초기화 → **전처리** → **추론** → **후처리** → 제어 → 액추에이터 → 파이프라인 루프

> **보드 전용 셀**에는 `# BOARD ONLY` 주석이 있습니다. 로컬 PC에서는 해당 셀을 건너뛰고 DryRun 모드로 테스트하세요.

---
## 1. 환경 설정 및 라이브러리 임포트

In [1]:
import os
import sys
import time
import threading
from pathlib import Path

import cv2
import numpy as np
import yaml
from postprocessing import RTLEquivalentPostProcessor

# ── setup.sh 환경변수 로드 (DPU 런타임에 필요) ────────────────────────────────
def load_env_script(filepath="setup.sh"):
    if os.path.exists(filepath):
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line.startswith("export "):
                    kv = line[7:].split("=", 1)
                    if len(kv) == 2:
                        os.environ[kv[0].strip()] = kv[1].strip()

load_env_script("setup.sh")

# ── pynq_dpu 오버레이 (보드 전용) ─────────────────────────────────────────────
try:
    from pynq_dpu import DpuOverlay  # type: ignore
    BOARD = True
    print("[OK] pynq_dpu 임포트 성공 → 보드 모드")
except Exception:
    DpuOverlay = None
    BOARD = False
    print("[INFO] pynq_dpu 없음 → DryRun(시뮬레이션) 모드")

print(f"Python {sys.version}")
print(f"OpenCV {cv2.__version__}")

[OK] pynq_dpu 임포트 성공 → 보드 모드
Python 3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
OpenCV 4.5.4


In [2]:
# ── 설정 파일 로드 ─────────────────────────────────────────────────────────────
CONFIG_DIR = "configs"

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

cfg = {
    "default": load_yaml(f"{CONFIG_DIR}/default.yaml"),
    "camera":  load_yaml(f"{CONFIG_DIR}/camera.yaml"),
    "model":   load_yaml(f"{CONFIG_DIR}/model.yaml"),
    "control": load_yaml(f"{CONFIG_DIR}/control.yaml"),
}

# 핵심 설정값 출력
print("=== 카메라 설정 ===")
print(f"  해상도: {cfg['camera']['width']}x{cfg['camera']['height']} @ {cfg['camera']['fps']}fps")
print(f"  ROI top ratio: {cfg['camera']['roi_top_ratio']}")
print()
print("=== 모델 설정 ===")
print(f"  xmodel: {cfg['model']['xmodel_path']}")
print(f"  입력 크기: {cfg['model']['input_width']}x{cfg['model']['input_height']}x{cfg['model']['input_channels']}")
print(f"  threshold: {cfg['model']['threshold']}")
print()
print("=== 제어 설정 ===")
print(f"  Kp: {cfg['control']['kp']}, base_speed: {cfg['control']['base_speed']}")
print(f"  use_actuator: {cfg['default'].get('use_actuator', False)}")

=== 카메라 설정 ===
  해상도: 640x480 @ 30fps
  ROI top ratio: 0.0

=== 모델 설정 ===
  xmodel: models/lane_segmentation.xmodel
  입력 크기: 256x256x3
  threshold: 0.0

=== 제어 설정 ===
  Kp: 0.2, base_speed: 0.25
  use_actuator: True


---
## 2. 전처리 (Preprocessing)

카메라 BGR 프레임 → DPU 입력 텐서 변환

**처리 순서:** ROI 크롭 → 리사이즈 → BGR→RGB 변환 → float32 정규화

In [3]:
# ── 전처리 함수 정의 ───────────────────────────────────────────────────────────

def _compute_roi(frame_bgr, cfg_camera):
    """ratio 값을 실제 픽셀 좌표로 변환"""
    h, w = frame_bgr.shape[:2]
    top    = max(0, min(int(h * cfg_camera["roi_top_ratio"]),    h - 1))
    bottom = max(top + 1, min(int(h * cfg_camera["roi_bottom_ratio"]), h))
    left   = max(0, min(int(w * cfg_camera["roi_left_ratio"]),   w - 1))
    right  = max(left + 1, min(int(w * cfg_camera["roi_right_ratio"]),  w))
    return top, bottom, left, right


def crop_roi(frame_bgr, camera_cfg):
    """관심 영역(ROI)만 잘라낸다. 하늘 등 불필요한 상단 영역 제거."""
    top, bottom, left, right = _compute_roi(frame_bgr, camera_cfg)
    roi_bgr = frame_bgr[top:bottom, left:right]
    meta = {
        "orig_h": frame_bgr.shape[0], "orig_w": frame_bgr.shape[1],
        "roi_top": top, "roi_bottom": bottom,
        "roi_left": left, "roi_right": right,
        "roi_h": roi_bgr.shape[0], "roi_w": roi_bgr.shape[1],
    }
    return roi_bgr, meta


def resize_frame(image, target_w, target_h):
    """DPU 입력 크기로 리사이즈 (INTER_LINEAR 양선형 보간)"""
    return cv2.resize(image, (target_w, target_h), interpolation=cv2.INTER_LINEAR)


def convert_color(image_bgr, channel_order="RGB"):
    """BGR → 모델 학습 채널 순서로 변환 (기본 RGB)"""
    if channel_order.upper() == "RGB":
        return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_bgr


def normalize_image(image, normalize=True, mean=None, std=None):
    """픽셀값을 float32로 변환 후 정규화. 순서: /255 → mean 빼기 → std 나누기"""
    out = image.astype(np.float32)
    if normalize:
        out *= (1.0 / 255.0)
    if mean is not None:
        out -= np.array(mean, dtype=np.float32)
    if std is not None:
        out /= np.array(std, dtype=np.float32)
    return out


def preprocess_frame(frame_bgr, camera_cfg, model_cfg):
    """
    카메라 프레임 → DPU 입력 텐서 전처리 파이프라인 전체 실행.
    Returns: (input_image [H,W,C float32], meta dict)
    """
    roi_bgr, meta   = crop_roi(frame_bgr, camera_cfg)
    resized         = resize_frame(roi_bgr, model_cfg["input_width"], model_cfg["input_height"])
    converted       = convert_color(resized, model_cfg.get("channel_order", "RGB"))
    input_image     = normalize_image(
        converted,
        normalize=model_cfg.get("normalize", True),
        mean=model_cfg.get("mean"),
        std=model_cfg.get("std"),
    )
    meta.update({
        "input_h": model_cfg["input_height"],
        "input_w": model_cfg["input_width"],
        "channel_order": model_cfg.get("channel_order", "RGB"),
    })
    return input_image, meta


print("전처리 함수 정의 완료")

전처리 함수 정의 완료


In [4]:
# ── 전처리 단계별 동작 확인 (더미 이미지로 테스트) ─────────────────────────────
dummy_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

roi_bgr, meta = crop_roi(dummy_frame, cfg["camera"])
print(f"[1] ROI 크롭: {dummy_frame.shape} → {roi_bgr.shape}")
print(f"    좌표: top={meta['roi_top']}, bottom={meta['roi_bottom']}, "
      f"left={meta['roi_left']}, right={meta['roi_right']}")

resized = resize_frame(roi_bgr, cfg["model"]["input_width"], cfg["model"]["input_height"])
print(f"[2] 리사이즈: {roi_bgr.shape} → {resized.shape}")

converted = convert_color(resized, cfg["model"].get("channel_order", "RGB"))
print(f"[3] 색상 변환 (BGR→{cfg['model'].get('channel_order','RGB')}): dtype={converted.dtype}")

normalized = normalize_image(converted, normalize=cfg["model"]["normalize"])
print(f"[4] 정규화: dtype={normalized.dtype}, min={normalized.min():.4f}, max={normalized.max():.4f}")

input_image, meta = preprocess_frame(dummy_frame, cfg["camera"], cfg["model"])
print(f"\n전처리 최종 출력: shape={input_image.shape}, dtype={input_image.dtype}")

[1] ROI 크롭: (480, 640, 3) → (480, 640, 3)
    좌표: top=0, bottom=480, left=0, right=640
[2] 리사이즈: (480, 640, 3) → (256, 256, 3)
[3] 색상 변환 (BGR→RGB): dtype=uint8
[4] 정규화: dtype=float32, min=0.0039, max=0.9882

전처리 최종 출력: shape=(256, 256, 3), dtype=float32


---
## 3. 추론 (DPU Inference)

xmodel 로드 → DPU subgraph 추출 → vart.Runner 생성 → float32 입력 → int8 양자화 → 추론 → float32 역양자화

> **보드 전용 셀입니다.** 로컬 PC에서는 더미 출력으로 대체됩니다.

In [5]:
# ── DPURunner 클래스 정의 ──────────────────────────────────────────────────────

class DPURunner:
    """
    xmodel 기반 DPU 추론 클래스.
    load_model() → create_runner() → run(image) 순서로 사용.
    보드(PYNQ + Vitis AI) 환경에서만 실제 추론 가능.
    """

    def __init__(self, model_cfg):
        self.model_cfg      = model_cfg
        self.graph          = None
        self.runner         = None
        self.input_tensors  = None
        self.output_tensors = None
        self._xir = self._vart = None
        # 사전 할당 버퍼 (create_runner 후 채워짐)
        self._in_f32 = self._in_buf = self._out_f32 = None
        self._out_buf = None
        self._in_scale = self._out_scale = 1.0

    def _lazy_import(self):
        try:
            import xir   # type: ignore
            import vart  # type: ignore
        except ImportError as e:
            raise RuntimeError("xir/vart import 실패. PYNQ + Vitis AI 환경 필요") from e
        self._xir, self._vart = xir, vart

    def load_model(self):
        """xmodel 파일 → xir.Graph 역직렬화"""
        self._lazy_import()
        xmodel_path = Path(self.model_cfg["xmodel_path"])
        if not xmodel_path.exists():
            raise FileNotFoundError(f"xmodel 없음: {xmodel_path}")
        self.graph = self._xir.Graph.deserialize(str(xmodel_path))
        print(f"[DPU] xmodel 로드 완료: {xmodel_path}")

    def _get_dpu_subgraph(self):
        root = self.graph.get_root_subgraph()
        for child in root.toposort_child_subgraph():
            if child.has_attr("device") and child.get_attr("device").upper() == "DPU":
                return child
        raise RuntimeError("DPU subgraph를 찾지 못함")

    def create_runner(self):
        """vart.Runner 생성 및 버퍼 사전 할당"""
        if self.graph is None:
            raise RuntimeError("load_model() 먼저 실행 필요")
        dpu_sg             = self._get_dpu_subgraph()
        # 모델 내 전체 subgraph 출력 — CPU subgraph가 있으면 현재 코드에서 무시됨
        root = self.graph.get_root_subgraph()
        for _child in root.toposort_child_subgraph():
            _dev = _child.get_attr("device") if _child.has_attr("device") else "UNKNOWN"
            _tag = " <- 사용" if _child is dpu_sg else ""
            print(f"  subgraph: {_child.get_name()}, device={_dev}{_tag}")
        self.runner        = self._vart.Runner.create_runner(dpu_sg, "run")
        self.input_tensors = self.runner.get_input_tensors()
        self.output_tensors= self.runner.get_output_tensors()

        in_t  = self.input_tensors[0]
        out_t = self.output_tensors[0]
        self._in_f32  = np.empty(tuple(in_t.dims),  dtype=np.float32)
        self._in_buf  = np.empty(tuple(in_t.dims),  dtype=np.int8)
        self._out_buf = [np.empty(tuple(t.dims), dtype=np.int8) for t in self.output_tensors]
        self._out_f32 = np.empty(tuple(out_t.dims), dtype=np.float32)

        def _fp(tensor):
            # has_attr 선행 확인 필수: fix_point 없는 텐서에서 get_attr()는
            # Python 예외가 아닌 C++ abort를 일으킴
            if tensor.has_attr("fix_point"):
                return tensor.get_attr("fix_point")
            return 0

        in_fp  = _fp(in_t)
        out_fp = _fp(out_t)
        self._in_scale  = float(2 **  in_fp)
        self._out_scale = float(2 ** -out_fp)
        print(f"[DPU] Runner 생성 완료")
        print(f"      입력 shape : {tuple(in_t.dims)}, in_scale={self._in_scale} (fix_point={in_fp})")
        print(f"      출력 shape : {tuple(out_t.dims)}, out_scale={self._out_scale} (fix_point={out_fp})")

    def get_input_shape(self):
        return tuple(self.input_tensors[0].dims) if self.input_tensors else ()

    def get_output_shape(self):
        return tuple(self.output_tensors[0].dims) if self.output_tensors else ()

    def run(self, image):
        """
        전처리된 이미지(H,W,C float32) → DPU 추론 → float32 출력.
        사전 할당 버퍼를 재사용해 매 프레임 메모리 할당 없음.
        """
        if self.runner is None:
            raise RuntimeError("create_runner() 먼저 실행 필요")
        # float32 → int8 양자화
        np.multiply(image, self._in_scale, out=self._in_f32[0])
        np.clip(self._in_f32[0], -128, 127, out=self._in_f32[0])
        np.copyto(self._in_buf[0], self._in_f32[0], casting="unsafe")
        # DPU 비동기 추론
        job_id = self.runner.execute_async([self._in_buf], self._out_buf)
        self.runner.wait(job_id)
        # int8 → float32 역양자화
        np.multiply(self._out_buf[0], self._out_scale, out=self._out_f32)
        return self._out_f32

    def warmup(self, image, num_iters=2):
        for _ in range(num_iters):
            self.run(image)


print("DPURunner 클래스 정의 완료")

DPURunner 클래스 정의 완료


In [6]:
# ── DPU 오버레이 로드 및 Runner 초기화 (BOARD ONLY) ───────────────────────────
dpu_runner = DPURunner(cfg["model"])

if BOARD:
    actuator_cfg = cfg["default"].get("actuator", {})
    overlay_bit  = str(Path(actuator_cfg.get("overlay_path", "configs/dpu/dpu.bit")).resolve())
    print(f"DPU overlay 로드: {overlay_bit}")
    _dpu_overlay = DpuOverlay(overlay_bit)
    print("DPU overlay 로드 완료")

    dpu_runner.load_model()
    dpu_runner.create_runner()

    # 워밍업
    warmup_iters = int(cfg["default"].get("warmup_iters", 2))
    _, in_h, in_w, in_c = dpu_runner.get_input_shape()
    print(f"DPU 워밍업 {warmup_iters}회...")
    dpu_runner.warmup(np.zeros((in_h, in_w, in_c), dtype=np.float32), warmup_iters)
    print("DPU 초기화 완료")
else:
    print("[DryRun] DPU 초기화 생략 → 더미 출력으로 대체됩니다")
    # 더미 run 함수 (로컬 테스트용)
    _H, _W = cfg["model"]["input_height"], cfg["model"]["input_width"]
    def _dummy_run(image):
        return np.zeros((1, _H, _W, 1), dtype=np.float32)
    dpu_runner.run = _dummy_run
    print(f"더미 출력 shape: (1, {_H}, {_W}, 1)")

DPU overlay 로드: /home/xilinx/jupyter_notebooks/pynq-dpu/auto_drive/configs/dpu/dpu.bit


DPU overlay 로드 완료
[DPU] xmodel 로드 완료: models/lane_segmentation.xmodel
  subgraph: subgraph_input_1, device=USER
  subgraph: subgraph_quant_add, device=DPU <- 사용
  subgraph: subgraph_quant_conv2d_51_fix_, device=CPU
[DPU] Runner 생성 완료
      입력 shape : (1, 256, 256, 3), in_scale=64.0 (fix_point=6)
      출력 shape : (1, 256, 256, 1), out_scale=0.25 (fix_point=2)
DPU 워밍업 2회...
DPU 초기화 완료


---
## 4. 후처리 (Postprocessing)

DPU float32 logits → binary 마스크 → 노이즈 제거 → ROI 복원 → 중심선 추출 → 기준점 선택 → 조향 오차 계산

In [7]:
# ── 후처리 함수 정의 ───────────────────────────────────────────────────────────

_morph_kernels = {}  # morphology 커널 캐시 (매 프레임 재생성 방지)

def _get_kernel(size):
    if size not in _morph_kernels:
        _morph_kernels[size] = np.ones((size, size), np.uint8)
    return _morph_kernels[size]


def logits_to_mask(raw_output, threshold=0.0):
    """
    DPU 출력 logits → binary 마스크 (0 or 255).
    threshold=0.0 기본값은 quantize_v3.py가 sigmoid를 제거하고 컴파일하기 때문.
    logit > 0.0 은 sigmoid(logit) > 0.5 와 동치 — 모델 변경 시 threshold 재검토 필요.
    shape 정규화: (1,H,W,1), (1,H,W), (H,W,1) 등 모두 처리.
    """
    arr = raw_output
    if arr.ndim == 4 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    elif arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim != 2:
        raise RuntimeError(f"지원하지 않는 output shape: {raw_output.shape}")
    mask = (arr > threshold).astype(np.uint8)
    mask *= 255
    return mask


def filter_noise(mask, kernel_size=5):
    """Morphology OPEN→CLOSE로 노이즈 제거 및 구멍 메우기"""
    kernel = _get_kernel(kernel_size)
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return closed


def filter_components(mask, min_area=0):
    """작은 connected component 제거 (min_area 픽셀 미만 blob 삭제)"""
    if min_area <= 0:
        return mask
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask
    keep = np.zeros(num_labels, dtype=np.uint8)
    keep[1:] = (stats[1:, cv2.CC_STAT_AREA] >= min_area).astype(np.uint8)
    return (keep[labels] * 255).astype(np.uint8)


def restore_mask_to_roi_size(mask, preprocess_meta):
    """DPU 출력 크기(예: 256x256) → 원래 ROI 크기로 복원 (INTER_NEAREST)"""
    return cv2.resize(
        mask,
        (preprocess_meta["roi_w"], preprocess_meta["roi_h"]),
        interpolation=cv2.INTER_NEAREST,
    )


def compute_centerline(mask_roi, row_step=5, min_pixels_per_row=1):
    """
    행(row)별 차선 픽셀 중심 x좌표 추출.
    numpy 벡터 연산으로 전체 행을 한 번에 처리.
    Returns: [(x, y), ...] (하단부터 상단 순)
    """
    h, w        = mask_roi.shape[:2]
    row_step    = max(1, int(row_step))
    min_pix     = max(1, int(min_pixels_per_row))
    ys          = np.arange(h - 1, -1, -row_step)
    rows        = mask_roi[ys] > 0
    row_sums    = rows.sum(axis=1)
    valid       = row_sums >= min_pix
    if not valid.any():
        return []
    xs          = np.arange(w, dtype=np.int32)
    x_sums      = (rows * xs).sum(axis=1)
    center_xs   = np.where(valid, x_sums // row_sums.clip(1), 0)
    return [(int(center_xs[i]), int(ys[i])) for i in range(len(ys)) if valid[i]]


def select_reference_point(centerline_points, mask_roi_shape, reference_row_ratio=0.7):
    """reference_row_ratio 높이에 가장 가까운 중심선 포인트를 기준점으로 선택"""
    h, _ = mask_roi_shape
    target_y = int(h * reference_row_ratio)
    if not centerline_points:
        return None, None
    return min(centerline_points, key=lambda p: abs(p[1] - target_y))


def compute_steering_error(ref_x, roi_w):
    """기준점 x좌표 → 정규화 조향 오차 (-1.0=좌, 0=중앙, 1.0=우)"""
    if ref_x is None:
        return 0.0
    return float((ref_x - roi_w / 2.0) / (roi_w / 2.0))


def compute_heading_error(centerline_points):
    """중심선 첫점~끝점 기울기로 헤딩 오차 계산 (dx/dy)"""
    if len(centerline_points) < 2:
        return 0.0
    p1, p2 = centerline_points[0], centerline_points[-1]
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]
    return 0.0 if dy == 0 else float(dx / dy)


def postprocess_output(raw_output, preprocess_meta, model_cfg, control_cfg):
    """
    DPU 출력 → 조향 제어 값 전체 계산 파이프라인.
    Returns dict: roi_mask, centerline_points, reference_point,
                  steering_error, heading_error, lane_pixels, valid
    """
    # 1. logits → binary 마스크
    mask     = logits_to_mask(raw_output, threshold=model_cfg["threshold"])
    # 2. 노이즈 제거
    filtered = filter_noise(mask, kernel_size=control_cfg["morph_kernel_size"])
    # 3. 작은 blob 제거
    filtered = filter_components(filtered, min_area=int(control_cfg.get("min_component_area", 0)))
    # 4. ROI 크기로 복원
    roi_mask = restore_mask_to_roi_size(filtered, preprocess_meta)
    # 5. 중심선 추출
    cl_pts   = compute_centerline(
        roi_mask,
        row_step=int(control_cfg.get("centerline_row_step", 5)),
        min_pixels_per_row=int(control_cfg.get("min_pixels_per_sample_row", 1)),
    )
    # 6. 기준점 선택
    ref_pt   = select_reference_point(
        cl_pts, roi_mask.shape[:2],
        reference_row_ratio=control_cfg["reference_row_ratio"],
    )
    # 7. 조향/헤딩 오차 계산
    steer_err  = compute_steering_error(ref_pt[0], preprocess_meta["roi_w"])
    heading_err= compute_heading_error(cl_pts)
    lane_pixels= int(np.count_nonzero(roi_mask))
    valid = (
        ref_pt[0] is not None
        and lane_pixels >= int(control_cfg.get("min_lane_pixels", 1))
        and len(cl_pts) >= int(control_cfg.get("min_centerline_points", 1))
    )
    return {
        "roi_mask":          roi_mask,
        "centerline_points": cl_pts,
        "reference_point":   ref_pt,
        "steering_error":    steer_err,
        "heading_error":     heading_err,
        "lane_pixels":       lane_pixels,
        "valid":             valid,
    }


print("후처리 함수 정의 완료")

후처리 함수 정의 완료


In [8]:
# ── 후처리 단계별 동작 확인 (더미 출력으로 테스트) ────────────────────────────
H, W = cfg["model"]["input_height"], cfg["model"]["input_width"]

# 배경은 음수, 중앙 차선만 양수인 int8 DPU 출력 패턴
dummy_logits = np.full((1, H, W, 1), -1, dtype=np.int8)
dummy_logits[0, :, W//3 : 2*W//3, 0] = 1
post = RTLEquivalentPostProcessor(cfg["model"], cfg["control"]).run(dummy_logits)

print(f"lane_pixels: {post['lane_pixels']}")
print(f"기준점: {post['reference_point']}")
print(f"steering_error: {post['steering_error']:+.4f}")
print(f"valid: {post['valid']}")

---
## 5. 제어기 (P Controller)

조향 오차 → P 제어 → steering_cmd / speed_cmd 계산

In [9]:
# ── PDController 클래스 정의 ────────────────────────────────────────────────────

def _clamp(v, lo, hi):
    return max(lo, min(hi, v))

class PDController:
    """
    비례-미분(PD) 제어기 + 로우패스 필터(EMA)
    """
    def __init__(self, cfg):
        self.kp           = cfg["kp"]
        self.kd           = cfg.get("kd", 0.5)  # YAML에서 kd를 못 찾으면 기본값 0.5 사용
        self.deadband     = cfg["deadband"]
        self.max_steering = cfg["max_steering"]
        self.min_steering = cfg["min_steering"]
        self.base_speed   = cfg["base_speed"]
        self.min_speed    = cfg["min_speed"]
        self.max_speed    = cfg["max_speed"]
        
        self.prev_error = 0.0          # D 제어를 위한 이전 오차
        self.prev_steering_cmd = 0.0   # 필터를 위한 이전 조향값
        self.alpha = 0.6               # EMA 필터 계수 (핸들 떨림 방지)

    def compute(self, steering_error):
        if abs(steering_error) < self.deadband:
            steering_error = 0.0
            
        # 1. 오차의 변화량(미분항) 계산
        error_diff = steering_error - self.prev_error
        self.prev_error = steering_error
            
        # 2. PD 제어 계산
        target_steering_cmd = (self.kp * steering_error) + (self.kd * error_diff)
        target_steering_cmd = _clamp(target_steering_cmd, self.min_steering, self.max_steering)
        
        # 3. 로우패스 필터 적용 (과거 값과 섞어서 부드럽게)
        steering_cmd = (self.alpha * target_steering_cmd) + ((1.0 - self.alpha) * self.prev_steering_cmd)
        self.prev_steering_cmd = steering_cmd

        speed_cmd = _clamp(
            self.base_speed * max(0.4, 1.0 - abs(steering_cmd)),
            self.min_speed, self.max_speed
        )
        return {"steering_cmd": float(steering_cmd), "speed_cmd": float(speed_cmd), "mode": "drive"}

# PController 대신 새로운 PDController로 교체!
controller = PDController(cfg["control"])

# 동작 확인 (더미 테스트)
for err in [-0.5, -0.01, 0.0, 0.3, 1.0]:
    cmd = controller.compute(err)
    print(f"  오차={err:+.2f} → steer={cmd['steering_cmd']:+.3f}, speed={cmd['speed_cmd']:.3f}")

  오차=-0.50 → steer=-0.210, speed=0.198
  오차=-0.01 → steer=+0.066, speed=0.234
  오차=+0.00 → steer=+0.026, speed=0.243
  오차=+0.30 → steer=+0.137, speed=0.216
  오차=+1.00 → steer=+0.385, speed=0.154


---
## 6. 액추에이터 (Actuator)

- **DryRunActuator**: 하드웨어 없이 명령 시뮬레이션 (로컬 테스트용)
- **PynqMMIOActuator**: FPGA PWM 레지스터 직접 제어 (보드 전용)

In [10]:
# ── 액추에이터 클래스 정의 ─────────────────────────────────────────────────────

# 보드 전용 라이브러리 지연 임포트
try:
    from pynq import MMIO  # type: ignore
except Exception:
    MMIO = None

try:
    import spidev  # type: ignore
except Exception:
    spidev = None

# FPGA 레지스터 오프셋
REG_PERIOD = 0x00  # PWM 주기 클럭 수
REG_DUTY   = 0x04  # PWM duty 클럭 수
REG_VALID  = 0x08  # 출력 enable (1=on)


class DryRunActuator:
    """하드웨어 없이 명령만 받아 내부 상태를 갱신하는 테스트용 액추에이터"""

    def __init__(self, cfg=None):
        self.cfg = cfg or {}
        self.initialized       = False
        self._current_steering = 0.0

    def initialize(self):
        self.initialized = True
        print("[DryRun] 액추에이터 초기화")

    def apply_control(self, control_command):
        if not self.initialized:
            raise RuntimeError("initialize() 먼저 호출 필요")
        steer = _clamp(float(control_command.get("steering_cmd", 0.0)), -1.0, 1.0)
        speed = _clamp(float(control_command.get("speed_cmd",    0.0)), -1.0, 1.0)
        mode  = control_command.get("mode", "drive")
        if mode == "stop":
            steer = speed = 0.0
        self._current_steering = steer
        return {
            "current_steering": self._current_steering,
            "adc_raw":          None,
            "applied_steering": steer,
            "applied_speed":    speed,
            "steering_effort":  steer,
            "mode":             "stop" if mode == "stop" else "dry_run",
        }

    def stop(self):
        self._current_steering = 0.0

    def close(self):
        self.stop()
        self.initialized = False


class PynqMMIOActuator:
    """pynq.MMIO로 FPGA PWM IP 레지스터를 직접 제어하는 액추에이터 (보드 전용)

    ★ 워치독 수정: threading.Timer를 매 프레임 생성/취소하던 방식을
      단일 워치독 스레드(_watchdog_loop)로 교체.
      Timer 방식은 프레임마다 OS 스레드를 생성해 리소스 누수 및
      Ultra96V2 hang의 원인이 됐음.
    """

    def __init__(self, cfg=None):
        self.cfg              = cfg or {}
        self.mmios            = {}
        self.spi              = None
        self.initialized      = False
        self._last_cmd_time   = 0.0               # 마지막 명령 시각 (워치독용)
        self._watchdog_stop   = threading.Event()  # 워치독 스레드 종료 신호
        self._watchdog_thread = None

    @staticmethod
    def _parse_addr(addr):
        return int(addr, 0) if isinstance(addr, str) else int(addr)

    def initialize(self):
        if MMIO is None:
            raise RuntimeError("pynq.MMIO import 실패. PYNQ 보드 환경 필요")
        motor_cfg   = self.cfg.get("motors", {})
        base_addrs  = motor_cfg.get("base_addrs", {})
        mmio_range  = int(self.cfg.get("mmio_range", 0x10000))
        if not base_addrs:
            raise RuntimeError("모터 주소 설정 없음")
        self.mmios  = {name: MMIO(self._parse_addr(addr), mmio_range) for name, addr in base_addrs.items()}
        self._period              = int(self.cfg.get("period_size",                  200000))
        self._drive_duty_pct      = float(self.cfg.get("drive_duty_percent",            1.0))
        self._steer_duty_pct      = float(self.cfg.get("steering_duty_percent",         1.0))
        self._steer_min_duty_pct  = float(self.cfg.get("steering_min_duty_percent",     0.0))
        self._drive_min_duty_pct  = float(self.cfg.get("drive_min_duty_percent",          0.0))
        center_pct                = float(self.cfg.get("steering_center_hold_percent",  0.0))
        self._steer_center_duty   = int(self._period * self._steer_duty_pct * center_pct)
        self._fwd_channels        = motor_cfg.get("drive_channels",   [])
        self._bwd_channels        = motor_cfg.get("reverse_channels", [])
        self._r_name              = motor_cfg.get("steering_right")
        self._l_name              = motor_cfg.get("steering_left")
        self._adc_left            = float(self.cfg.get("adc_left",   990))
        self._adc_right           = float(self.cfg.get("adc_right",  240))
        self._steer_feedback      = bool(self.cfg.get("steering_feedback_control", True))
        self._steer_kp            = float(self.cfg.get("steering_feedback_kp", 1.0))
        self._steer_deadband      = float(self.cfg.get("steering_feedback_deadband", 0.05))
        self._command_timeout_sec = float(self.cfg.get("command_timeout_sec", 0.3))
        self._debug_readback      = bool(self.cfg.get("debug_readback", False))
        for name in self.mmios:
            self.mmios[name].write(REG_PERIOD, self._period)
            self.mmios[name].write(REG_DUTY,   self._period)
            self.mmios[name].write(REG_VALID,  0)
        if self.cfg.get("enable_adc", True):
            if spidev is None:
                raise RuntimeError("spidev import 실패")
            self.spi = spidev.SpiDev()
            self.spi.open(int(self.cfg.get("spi_bus", 0)), int(self.cfg.get("spi_device", 0)))
            self.spi.max_speed_hz = int(self.cfg.get("spi_max_speed_hz", 20_000_000))
            self.spi.mode = int(self.cfg.get("spi_mode", 0))
        self.initialized = True

        # 워치독 스레드 시작 (단일 스레드, 매 프레임 Timer 생성 없음)
        self._last_cmd_time = time.time()
        self._watchdog_stop.clear()
        self._watchdog_thread = threading.Thread(target=self._watchdog_loop, daemon=True)
        self._watchdog_thread.start()

        print(f"[MMIO] 액추에이터 초기화 완료 (Period: {self._period}, "
              f"drive_min_duty: {self._drive_min_duty_pct:.0%}, "
              f"steer_min_duty: {self._steer_min_duty_pct:.0%}, "
              f"center_hold_duty: {center_pct:.0%})")

    def _watchdog_loop(self):
        """단일 워치독 스레드 — 명령 타임아웃 시 PWM 차단.

        200ms 주기로 마지막 명령 시각을 확인.
        threading.Timer 방식과 달리 스레드를 매 프레임 생성하지 않음.
        """
        timeout = self._command_timeout_sec
        while not self._watchdog_stop.is_set():
            self._watchdog_stop.wait(timeout=0.2)   # 200ms 대기 (Event로 즉시 깨울 수 있음)
            if self._watchdog_stop.is_set():
                break
            if timeout > 0 and (time.time() - self._last_cmd_time) > timeout:
                print("[Watchdog] 명령 타임아웃 → 모터 정지")
                self.stop()
                break

    def _write_duty(self, name, value):
        if name in self.mmios:
            self.mmios[name].write(REG_DUTY, int(value))

    def _write_valid(self, name, enable):
        if name in self.mmios:
            if enable:
                self.mmios[name].write(REG_PERIOD, self._period)
            self.mmios[name].write(REG_VALID, 1 if enable else 0)

    def _scaled_duty(self, cmd_abs, duty_pct):
        return int(self._period * duty_pct * _clamp(abs(cmd_abs), 0.0, 1.0))

    def _steer_duty(self, effort_abs):
        """DC 모터 최소 기동 듀티 보장: effort를 [min_duty, 1.0] 범위로 리매핑"""
        min_pct   = self._steer_min_duty_pct
        effective = min_pct + (1.0 - min_pct) * _clamp(effort_abs, 0.0, 1.0)
        return int(self._period * self._steer_duty_pct * effective)

    def _drive_duty(self, speed_abs):
        """DC 구동 모터 최소 기동 듀티 보장: speed를 [min_duty, 1.0] 범위로 리매핑"""
        min_pct   = self._drive_min_duty_pct
        effective = min_pct + (1.0 - min_pct) * _clamp(speed_abs, 0.0, 1.0)
        return int(self._period * self._drive_duty_pct * effective)

    def _read_regs(self, names):
        return {
            name: {"period": int(self.mmios[name].read(REG_PERIOD)),
                   "duty":   int(self.mmios[name].read(REG_DUTY)),
                   "valid":  int(self.mmios[name].read(REG_VALID))}
            for name in names if name in self.mmios
        }

    def read_adc_raw(self):
        if self.spi is None:
            return None
        r = self.spi.xfer2([0x00, 0x00])
        return int(((r[0] & 0x0F) << 8) | r[1])

    def _raw_to_steering(self, raw):
        if raw is None:
            return 0.0
        lo, hi = self._adc_right, self._adc_left
        if lo == hi:
            return 0.0
        pos_01 = _clamp((raw - lo) / (hi - lo), 0.0, 1.0)
        return 1.0 - pos_01 * 2.0

    def _apply_drive(self, speed_cmd):
        duty = self._drive_duty(abs(speed_cmd))
        for name in self._fwd_channels + self._bwd_channels:
            self._write_valid(name, False)
        if abs(speed_cmd) < 0.05:
            return
        for name in (self._fwd_channels if speed_cmd > 0 else self._bwd_channels):
            self._write_duty(name, duty)
            self._write_valid(name, True)

    def _compute_steering_effort(self, steer_cmd, cur_steer):
        effort = (steer_cmd - cur_steer) * self._steer_kp if (self._steer_feedback and cur_steer is not None) else steer_cmd
        effort = _clamp(effort, -1.0, 1.0)
        return 0.0 if abs(effort) < self._steer_deadband else effort

    def _apply_steering(self, effort):
        if abs(effort) < self._steer_deadband:
            # 중앙 유지: 양쪽 채널 동일 duty → 기구적 중앙으로 복귀력 제공
            if self._steer_center_duty > 0:
                self._write_duty(self._r_name, self._steer_center_duty)
                self._write_duty(self._l_name, self._steer_center_duty)
                self._write_valid(self._r_name, True)
                self._write_valid(self._l_name, True)
            else:
                self._write_valid(self._r_name, False)
                self._write_valid(self._l_name, False)
            return

        # 방향 조향: 한쪽만 활성화, 반대쪽 비활성
        self._write_valid(self._r_name, False)
        self._write_valid(self._l_name, False)
        duty = self._steer_duty(abs(effort))
        name = self._r_name if effort > 0.0 else self._l_name
        self._write_duty(name, duty)
        self._write_valid(name, True)

    def apply_control(self, control_command):
        if not self.initialized:
            raise RuntimeError("initialize() 먼저 호출 필요")
        steer_cmd = _clamp(float(control_command.get("steering_cmd", 0.0)), -1.0, 1.0)
        speed_cmd = _clamp(float(control_command.get("speed_cmd",    0.0)), -1.0, 1.0)
        mode      = control_command.get("mode", "drive")
        adc_raw   = self.read_adc_raw()
        cur_steer = self._raw_to_steering(adc_raw) if adc_raw is not None else None
        effort = drive_duty = steer_duty = 0
        act_drive_ch = []
        act_steer_ch = None
        regs = {}
        if mode == "stop":
            self.stop()
            steer_cmd = speed_cmd = 0.0
        else:
            effort = self._compute_steering_effort(steer_cmd, cur_steer)
            if abs(speed_cmd) >= 0.05:
                drive_duty   = self._drive_duty(abs(speed_cmd))
                act_drive_ch = list(self._fwd_channels if speed_cmd > 0 else self._bwd_channels)
            if abs(effort) >= self._steer_deadband:
                steer_duty   = self._steer_duty(abs(effort))
                act_steer_ch = self._r_name if effort > 0 else self._l_name
            else:
                steer_duty   = self._steer_center_duty  # 중앙 유지 duty 로깅
                act_steer_ch = "center"
            self._apply_drive(speed_cmd)
            self._apply_steering(effort)
            self._last_cmd_time = time.time()   # 워치독 갱신 (Timer 생성 없음)
            if self._debug_readback:
                names = act_drive_ch + (
                    [self._r_name, self._l_name] if act_steer_ch == "center"
                    else ([act_steer_ch] if act_steer_ch else [])
                )
                regs = self._read_regs(names)
        return {
            "current_steering": cur_steer, "adc_raw": adc_raw,
            "applied_steering": steer_cmd, "applied_speed": speed_cmd,
            "steering_effort":  effort,
            "drive_duty":        drive_duty, "drive_channels":    act_drive_ch,
            "steering_duty":     steer_duty, "steering_channel":  act_steer_ch,
            "registers":         regs,
            "mode":             mode if mode == "stop" else "real_mmio",
        }

    def stop(self):
        """모든 PWM 채널 비활성화."""
        for name in self.mmios:
            self._write_valid(name, False)

    def close(self):
        self._watchdog_stop.set()   # 워치독 스레드에 종료 신호 → 즉시 깨어나 종료
        try:
            self.stop()
        finally:
            if self.spi:
                self.spi.close()
            self.initialized = False


print("액추에이터 클래스 정의 완료")


액추에이터 클래스 정의 완료


In [11]:
# ── 액추에이터 인스턴스 생성 ───────────────────────────────────────────────────
actuator_cfg = cfg["default"].get("actuator", {})
use_actuator = cfg["default"].get("use_actuator", False) and BOARD

if use_actuator:
    actuator = PynqMMIOActuator(actuator_cfg)
    print("[MMIO] PynqMMIOActuator 사용")
else:
    actuator = DryRunActuator(actuator_cfg)
    print("[DryRun] DryRunActuator 사용")

actuator.initialize()

[MMIO] PynqMMIOActuator 사용
[MMIO] 액추에이터 초기화 완료 (Period: 600600, drive_min_duty: 60%, steer_min_duty: 40%, center_hold_duty: 30%)


---
## 7. 카메라 초기화

In [12]:
# ── CameraManager 클래스 정의 (비동기 캡처 버전) ──────────────────────────────
import threading
import queue

class CameraManager:
    """
    OpenCV VideoCapture 래퍼.
    별도 캡처 스레드가 계속 프레임을 읽어 큐에 쌓아두고,
    메인 루프는 대기 없이 최신 프레임을 가져간다.
    """

    def __init__(self, cfg):
        self.cfg = cfg
        self.cap = None
        self._queue = queue.Queue(maxsize=2)   # 최대 2프레임만 버퍼링
        self._stop_event = threading.Event()
        self._thread = None

    def open(self):
        camera_index = int(self.cfg["camera_index"])
        device_path  = Path(f"/dev/video{camera_index}")
        if not device_path.exists():
            available = sorted(str(p) for p in Path("/dev").glob("video*"))
            raise RuntimeError(
                f"카메라 장치 없음: {device_path} "
                f"(사용 가능: {', '.join(available) or '없음'})"
            )
        self.cap = cv2.VideoCapture(camera_index, cv2.CAP_V4L2)
        if not self.cap.isOpened():
            raise RuntimeError(f"카메라 열기 실패: index={camera_index}")

        pf = self.cfg.get("pixel_format")
        if pf:
            self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*str(pf).upper()[:4]))
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  self.cfg["width"])
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.cfg["height"])
        self.cap.set(cv2.CAP_PROP_FPS,          self.cfg["fps"])
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)  # 버퍼 최소화 → 항상 최신 프레임

        w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"[Camera] 열기 완료: index={camera_index}, 실제 해상도 {w}x{h}")

        # 캡처 스레드 시작
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._capture_loop, daemon=True)
        self._thread.start()
        print("[Camera] 캡처 스레드 시작")

    def _capture_loop(self):
        """캡처 전담 스레드 — 항상 cap.read() 호출로 V4L2 커널 버퍼를 소비한다.

        ★ 수정 이유: 큐가 꽉 찼을 때 cap.read()를 skip(continue)하면
          V4L2 드라이버 내부 DMA 버퍼가 소비되지 않아 커널 버퍼 포화 → hang 발생.
          해결: 항상 read 후 큐가 꽉 찼으면 오래된 프레임을 버리고 최신으로 교체.
        """
        while not self._stop_event.is_set():
            ret, frame = self.cap.read()   # 반드시 매 루프 호출 — skip 금지
            if not ret or frame is None:
                time.sleep(0.005)
                continue
            # 큐가 꽉 찼으면 오래된 프레임을 버리고 최신 프레임으로 교체
            if self._queue.full():
                try:
                    self._queue.get_nowait()   # 오래된 프레임 제거
                except queue.Empty:
                    pass
            try:
                self._queue.put_nowait((ret, frame))
            except queue.Full:
                pass   # 극히 드문 경합 상황 — 이번 프레임 skip

    def read(self):
        """메인 루프에서 호출 — 큐에서 최신 프레임을 꺼냄"""
        if self._thread is None:
            raise RuntimeError("open() 먼저 호출 필요")
        try:
            return self._queue.get(timeout=0.5)  # 0.5초 안에 프레임 없으면 실패
        except queue.Empty:
            return False, None

    def is_opened(self):
        return self.cap is not None and self.cap.isOpened()

    def release(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
            self._thread = None
        if self.cap is not None:
            self.cap.release()
            self.cap = None
        print("[Camera] 해제 완료")


# ── 카메라 초기화 ──────────────────────────────────────────────────────────────
camera = CameraManager(cfg["camera"])
camera.open()


[Camera] 열기 완료: index=0, 실제 해상도 640x480
[Camera] 캡처 스레드 시작


---
## 8. 자율주행 파이프라인 루프

캡처 → **전처리** → **추론** → **후처리** → 제어 → 액추에이터 순으로 매 프레임 실행

In [13]:
import time
import collections

def elapsed_ms(start: float) -> float:
    return (time.time() - start) * 1000

# ── 사전 할당 버퍼 (매 프레임 numpy 메모리 할당 제거) ──────────────────────────
_in_h  = cfg["model"]["input_height"]   # 256
_in_w  = cfg["model"]["input_width"]    # 256
_cam_h = cfg["camera"]["height"]        # 480
_cam_w = cfg["camera"]["width"]         # 640

# 전처리 버퍼: 매 프레임 새 배열 생성 대신 1회 할당 후 재사용
_buf_resized   = np.empty((_in_h, _in_w, 3), dtype=np.uint8)    # 192 KB
_buf_converted = np.empty((_in_h, _in_w, 3), dtype=np.uint8)    # 192 KB
_buf_norm      = np.empty((_in_h, _in_w, 3), dtype=np.float32)  # 768 KB

# ROI 좌표 사전 계산 (카메라 config 고정 → 루프 외부에서 1회)
_roi_top    = max(0, min(int(_cam_h * cfg["camera"]["roi_top_ratio"]),    _cam_h - 1))
_roi_bottom = max(_roi_top  + 1, min(int(_cam_h * cfg["camera"]["roi_bottom_ratio"]), _cam_h))
_roi_left   = max(0, min(int(_cam_w * cfg["camera"]["roi_left_ratio"]),   _cam_w - 1))
_roi_right  = max(_roi_left + 1, min(int(_cam_w * cfg["camera"]["roi_right_ratio"]),  _cam_w))
_roi_h      = _roi_bottom - _roi_top
_roi_w      = _roi_right  - _roi_left
# 정규화 스케일/오프셋 사전 계산
_norm_scale = np.float32(1.0 / 255.0) if cfg["model"].get("normalize", True) else np.float32(1.0)
_norm_mean  = np.array(cfg["model"].get("mean") or [0.0, 0.0, 0.0], dtype=np.float32)
_norm_std   = np.array(cfg["model"].get("std")  or [1.0, 1.0, 1.0], dtype=np.float32)
_need_mean  = not np.all(_norm_mean == 0.0)
_need_std   = not np.all(_norm_std  == 1.0)

# RTL postproc_top과 동일한 CPU golden model을 실제 SW 후처리로 사용
software_postproc = RTLEquivalentPostProcessor(cfg["model"], cfg["control"])
_pre_buf_bytes = _buf_resized.nbytes + _buf_converted.nbytes + _buf_norm.nbytes
print(f"사전 할당 전처리 버퍼 합계: {_pre_buf_bytes / 1024:.0f} KB")
print("후처리: postproc_top RTL-equivalent CPU 구현")

# ── 파이프라인 루프 설정 ───────────────────────────────────────────────────────
max_frames     = int(cfg["default"].get("max_frames", 200))
target_fps     = float(cfg["default"].get("target_fps", 10))
print_every    = max(1, int(cfg["default"].get("print_every_n_frames", 100)))
frame_interval = 1.0 / target_fps if target_fps > 0 else 0.0

stop_on_invalid     = cfg["default"].get("stop_on_invalid_mask", False)
invalid_stop_frames = int(cfg["default"].get("invalid_stop_frames", 10))
invalid_count       = 0

prev_steer  = 0.0
prev_speed  = cfg["control"]["base_speed"]
t_prev_loop = None

# ── 실시간 통계 누적기 (frame_logs 전체 저장 대신 O(1) 메모리) ────────────────
# 1000프레임 기준 기존 방식: ~2MB Python dict 누적
# 변경 후: running min/max/sum + deque(maxlen=100) → 고정 ~40KB
_STAT_KEYS = [
    "t_capture_ms",
    "t_pre_roi_ms", "t_pre_resize_ms", "t_pre_color_ms", "t_pre_norm_ms",
    "t_preprocess_ms",
    "t_inference_ms",
    "t_postprocess_ms",
    "t_control_ms", "t_actuator_ms", "t_total_ms",
]
_stats_sum = {k: 0.0          for k in _STAT_KEYS}
_stats_min = {k: float("inf") for k in _STAT_KEYS}
_stats_max = {k: float("-inf")for k in _STAT_KEYS}
_stats_n   = 0
_valid_n   = 0
frame_logs = collections.deque(maxlen=100)   # 최근 100프레임만 보관 (진단용)

print(f"파이프라인 시작: max_frames={max_frames}, target_fps={target_fps}")
print(f"stop_on_invalid={stop_on_invalid}, invalid_stop_frames={invalid_stop_frames}")
print("Ctrl+C로 중지")

for frame_id in range(max_frames if max_frames > 0 else 10**9):
    t_loop_start = time.time()

    # ── fps 계산 ──────────────────────────────────────────────────────
    if t_prev_loop is None:
        fps = 0.0
    else:
        fps = 1.0 / max(t_loop_start - t_prev_loop, 1e-6)
    t_prev_loop = t_loop_start

    # ── 1. 캡처 ───────────────────────────────────────────────────────
    t = time.time()
    ret, frame_bgr = camera.read()
    if not ret or frame_bgr is None:
        continue
    t_cap_ms = elapsed_ms(t)

    # ── 2. 전처리 (사전 할당 버퍼 + dst= 파라미터 → 중간 배열 생성 없음) ─
    t_pre_start = time.time()

    t = time.time()
    roi_bgr = frame_bgr[_roi_top:_roi_bottom, _roi_left:_roi_right]  # 뷰 — 복사 없음
    t_roi_ms = elapsed_ms(t)

    t = time.time()
    cv2.resize(roi_bgr, (_in_w, _in_h), dst=_buf_resized, interpolation=cv2.INTER_LINEAR)
    t_resize_ms = elapsed_ms(t)

    t = time.time()
    cv2.cvtColor(_buf_resized, cv2.COLOR_BGR2RGB, dst=_buf_converted)
    t_color_ms = elapsed_ms(t)

    t = time.time()
    np.multiply(_buf_converted, _norm_scale, out=_buf_norm)
    if _need_mean: np.subtract(_buf_norm, _norm_mean, out=_buf_norm)
    if _need_std:  np.divide(_buf_norm, _norm_std, out=_buf_norm)
    t_norm_ms = elapsed_ms(t)

    t_pre_ms = elapsed_ms(t_pre_start)

    # ── 3. 추론 ───────────────────────────────────────────────────────
    t = time.time()
    raw_output = dpu_runner.run(_buf_norm)
    t_inf_ms = elapsed_ms(t)

    # ── 4. 후처리 (postproc_top과 동일한 CPU golden model) ─────────────
    t_post_start = time.time()
    post = software_postproc.run(raw_output)
    t_post_ms = elapsed_ms(t_post_start)
    steer_err   = post["steering_error"]
    heading_err = post["heading_error"]
    lane_pixels = post["lane_pixels"]
    valid       = post["valid"]

    # ── 5. 제어 ───────────────────────────────────────────────────────
    t = time.time()
    if valid:
        invalid_count = 0
        ctrl = controller.compute(steer_err)
        prev_steer = ctrl["steering_cmd"]
        prev_speed = ctrl["speed_cmd"]
    else:
        invalid_count += 1
        ctrl = {"steering_cmd": 0.0, "speed_cmd": 0.0, "mode": "stop"}
        if stop_on_invalid and invalid_count >= invalid_stop_frames:
            print(f"[STOP] {invalid_count}프레임 연속 차선 미검출 → 비상 정지")
            actuator.apply_control(ctrl)
            break
    t_ctrl_ms = elapsed_ms(t)

    # ── 6. 액추에이터 ─────────────────────────────────────────────────
    t = time.time()
    act = actuator.apply_control(ctrl)
    t_act_ms = elapsed_ms(t)

    t_total_ms = elapsed_ms(t_loop_start)

    # ── 통계 누적 (O(1) 메모리) + 최근 프레임 진단 로그 ─────────────────
    _frame_data = {
        "frame_id":          frame_id,
        "t_capture_ms":      round(t_cap_ms,     3),
        "t_pre_roi_ms":      round(t_roi_ms,     3),
        "t_pre_resize_ms":   round(t_resize_ms,  3),
        "t_pre_color_ms":    round(t_color_ms,   3),
        "t_pre_norm_ms":     round(t_norm_ms,    3),
        "t_preprocess_ms":   round(t_pre_ms,     3),
        "t_inference_ms":    round(t_inf_ms,     3),
        "t_postprocess_ms":  round(t_post_ms,    3),
        "t_control_ms":      round(t_ctrl_ms,    3),
        "t_actuator_ms":     round(t_act_ms,     3),
        "t_total_ms":        round(t_total_ms,   3),
        "fps":               round(fps,           2),
        "valid":             valid,
        "invalid_count":     invalid_count,
        "lane_pixels":       lane_pixels,
        "steering_error":    round(steer_err,            6),
        "steering_cmd":      round(ctrl["steering_cmd"], 6),
        "speed_cmd":         round(ctrl["speed_cmd"],    6),
    }
    _stats_n += 1
    if valid: _valid_n += 1
    for _k in _STAT_KEYS:
        _v = _frame_data[_k]
        _stats_sum[_k] += _v
        if _v < _stats_min[_k]: _stats_min[_k] = _v
        if _v > _stats_max[_k]: _stats_max[_k] = _v
    frame_logs.append(_frame_data)   # deque(maxlen=100): 초과 시 자동 폐기

    if frame_id % print_every == 0:
        print(
            f"[frame {frame_id:04d}] "
            f"capture={t_cap_ms:.1f}ms | "
            f"preprocess={t_pre_ms:.1f}ms(roi_crop={t_roi_ms:.1f} resize={t_resize_ms:.1f} "
            f"color={t_color_ms:.1f} normalize={t_norm_ms:.1f}) | "
            f"inference={t_inf_ms:.1f}ms | "
            f"postprocess_SW_RTL_equiv={t_post_ms:.1f}ms | "
            f"total={t_total_ms:.1f}ms  fps={fps:.1f}  invalid={invalid_count}"
        )

    # ── FPS 제한 (target_fps=0이면 완전 스킵) ─────────────────────────
    if frame_interval > 0:
        spare = frame_interval - (time.time() - t_loop_start)
        if spare > 0.001:
            time.sleep(spare)

print(f"\n완료: {_stats_n}프레임 처리")


사전 할당 버퍼 합계: 1708 KB (절감: 매 프레임 ~1708 KB 할당압 제거)
파이프라인 시작: max_frames=1000, target_fps=0.0
stop_on_invalid=True, invalid_stop_frames=10
Ctrl+C로 중지
[frame 0000] capture=0.1ms | preprocess=5.7ms(roi_crop=0.0 resize=4.5 color=0.2 normalize=0.9) | inference=14.7ms | postprocess=8.5ms(mask=0.9 denoise=1.1 blob_filter=2.9 restore=0.8 centerline=2.3 ref_point=0.5) | total=29.6ms  fps=0.0  invalid=0
[frame 0010] capture=102.8ms | preprocess=5.6ms(roi_crop=0.0 resize=4.5 color=0.2 normalize=0.9) | inference=18.4ms | postprocess=7.7ms(mask=0.8 denoise=1.2 blob_filter=2.2 restore=0.8 centerline=2.2 ref_point=0.5) | total=134.9ms  fps=7.3  invalid=0
[frame 0020] capture=99.6ms | preprocess=5.6ms(roi_crop=0.0 resize=4.5 color=0.2 normalize=0.9) | inference=18.2ms | postprocess=7.6ms(mask=0.8 denoise=1.1 blob_filter=2.2 restore=0.8 centerline=2.2 ref_point=0.5) | total=131.6ms  fps=7.6  invalid=0
[frame 0030] capture=102.3ms | preprocess=5.7ms(roi_crop=0.0 resize=4.5 color=0.2 normalize=0.9) | infer

KeyboardInterrupt: 

---
## 9. 정리 및 프로파일링 요약

In [ ]:

# ── 자원 해제 ──────────────────────────────────────────────────────────────────
actuator.close()
camera.release()
print("자원 해제 완료")


In [ ]:
# ── 프로파일링 요약 (전체 실행 기준 running stats) ─────────────────────────────
if _stats_n > 0:
    indent_keys = {k for k in _STAT_KEYS if k.startswith("t_pre_") or k.startswith("t_post_")}
    print(f"\n{'단계':<26} {'평균(ms)':>10} {'최소(ms)':>10} {'최대(ms)':>10}")
    print("-" * 60)
    for key in _STAT_KEYS:
        prefix = "  " if key in indent_keys else ""
        avg = _stats_sum[key] / _stats_n
        print(f"{prefix}{key:<26} {avg:>10.2f} {_stats_min[key]:>10.2f} {_stats_max[key]:>10.2f}")
    print(f"\n유효 프레임: {_valid_n}/{_stats_n} ({100*_valid_n/_stats_n:.1f}%)")
    print(f"(※ 통계는 전체 {_stats_n}프레임 기준 / frame_logs는 최근 {len(frame_logs)}프레임만 보관)")
else:
    print("처리된 프레임이 없습니다.")
